# RTMW3D extraction on the DUTAI V100 server

Notebook adapted from Kaggle to run on the server with Python 3.11 and an NVIDIA V100. It reads the existing VideoBadminton clips and writes resumable `.npy` outputs under `processed/rtmw3d/`.

## One-time server setup

Run these commands in the server terminal from the project directory. Then select the kernel **Python 3.11 (RTMW3D)** for this notebook.

```bash
cd /home/dutai/khacbinh/badminton-stroke
uv venv --python 3.11 .venv_rtmw3d
source .venv_rtmw3d/bin/activate
python -m pip install --upgrade pip
python -m pip install "numpy<2"
python -m pip install torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
python -m pip install "setuptools<81" wheel openmim
mim install mmengine==0.10.7
mim install mmcv==2.1.0
mim install mmdet==3.3.0
git clone --depth 1 --branch v1.3.2 https://github.com/open-mmlab/mmpose.git /tmp/mmpose-v1.3.2
python -m pip install -e /tmp/mmpose-v1.3.2
python -m pip install ipykernel
python -m ipykernel install --user --name rtmw3d --display-name "Python 3.11 (RTMW3D)"
```

The dependency environment is separate from the project's regular `.venv`. The extraction cell defaults to a 50-video pilot; after checking its output, set `MAX_VIDEOS = None` to process the full dataset. Existing valid outputs are skipped when rerunning.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Locate the project root even if Jupyter starts in the notebooks/ folder.
PROJECT_ROOT = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "processed").exists():
        PROJECT_ROOT = candidate
        break
assert PROJECT_ROOT is not None, f"Không tìm thấy project root từ {Path.cwd()}"

# Override this environment variable if the dataset is stored elsewhere on the server.
dataset_override = os.environ.get("BADMINTON_VIDEO_ROOT")
dataset_candidates = [
    Path(dataset_override) if dataset_override else None,
    PROJECT_ROOT / "data" / "data" / "VideoBadminton_Dataset",
    PROJECT_ROOT / "data" / "VideoBadminton_Dataset",
    Path("/home/dutai/khacbinh/pbl5_doantotnghiep/data/data/VideoBadminton_Dataset"),
]
VIDEO_ROOT = next((p for p in dataset_candidates if p is not None and p.is_dir()), None)
assert VIDEO_ROOT is not None, (
    "Không thấy VideoBadminton_Dataset. Đặt biến BADMINTON_VIDEO_ROOT "
    "hoặc cập nhật dataset_candidates."
)

OUTPUT_ROOT = PROJECT_ROOT / "processed" / "rtmw3d"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
video_paths = sorted(VIDEO_ROOT.rglob("*.mp4"))

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("Project:", PROJECT_ROOT)
print("Dataset:", VIDEO_ROOT)
print("Số video tìm thấy:", len(video_paths))
print("Output:", OUTPUT_ROOT)
for path in video_paths[:5]:
    print(path)
assert video_paths, "Không tìm thấy file .mp4 trong dataset."


In [ ]:
import sys
from pathlib import Path
import torch
import mmcv
import mmengine
import mmdet
import mmpose

assert sys.version_info[:2] == (3, 11), (
    f"Kernel cần Python 3.11, hiện tại là {sys.version.split()[0]}. "
    "Chọn kernel Python 3.11 (RTMW3D)."
)
assert torch.cuda.is_available(), "Không thấy CUDA; kiểm tra GPU V100 và kernel đã chọn."

MMPose_ROOT = Path("/tmp/mmpose-v1.3.2")
PROJECT_3D = MMPose_ROOT / "projects" / "rtmpose3d"
assert (PROJECT_3D / "demo" / "rtmdet_m_640-8xb32_coco-person.py").exists(), (
    f"Thiếu MMPose source tại {MMPose_ROOT}. Chạy phần One-time server setup."
)
sys.path.insert(0, str(MMPose_ROOT))
sys.path.insert(0, str(PROJECT_3D))

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("MMCV:", mmcv.__version__)
print("MMEngine:", mmengine.__version__)
print("MMDetection:", mmdet.__version__)
print("MMPose:", mmpose.__version__)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from mmdet.apis import init_detector, inference_detector
from mmpose.apis import init_model, inference_topdown
from mmpose.utils import adapt_mmdet_pipeline

DET_CONFIG = PROJECT_3D / "demo" / "rtmdet_m_640-8xb32_coco-person.py"
POSE_CONFIG = (
    PROJECT_3D
    / "configs"
    / "rtmw3d-l_8xb64_cocktail14-384x288.py"
)

DET_CHECKPOINT = (
    "https://download.openmmlab.com/mmpose/v1/projects/rtmpose/"
    "rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth"
)
POSE_CHECKPOINT = (
    "https://download.openmmlab.com/mmpose/v1/wholebody_3d_keypoint/"
    "rtmw3d/rtmw3d-l_8xb64_cocktail14-384x288-"
    "794dbc78_20240626.pth"
)

# Khởi tạo model một lần, dùng lại cho tất cả video.
detector = init_detector(
    str(DET_CONFIG),
    DET_CHECKPOINT,
    device="cuda:0",
)
detector.cfg = adapt_mmdet_pipeline(detector.cfg)

pose_model = init_model(
    str(POSE_CONFIG),
    POSE_CHECKPOINT,
    device="cuda:0",
)
pose_model.cfg.model.test_cfg.mode = "vis"

print("RTMW3D và person detector đã sẵn sàng trên V100.")


In [ ]:
import csv
import os
import time

OUTPUT_ROOT = Path("/kaggle/working/rtmw3d_outputs")
STATUS_PATH = OUTPUT_ROOT / "status.csv"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Lần đầu giữ 50 để đo tốc độ và kiểm tra kết quả.
# Khi thử ổn, đổi thành None để chạy hết dataset.
MAX_VIDEOS = 50

videos_to_run = video_paths[:MAX_VIDEOS] if MAX_VIDEOS else video_paths
print(f"Sẽ xử lý {len(videos_to_run)} / {len(video_paths)} video")


def save_npy_atomic(path, array):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(path.stem + ".tmp.npy")
    np.save(temp_path, array)
    os.replace(temp_path, path)


status_exists = STATUS_PATH.exists()

with open(STATUS_PATH, "a", newline="") as status_file:
    writer = csv.DictWriter(
        status_file,
        fieldnames=["video", "status", "frames", "error"],
    )
    if not status_exists:
        writer.writeheader()

    started = time.time()

    for index, video_path in enumerate(videos_to_run, start=1):
        class_name = video_path.parent.name
        joints_path = OUTPUT_ROOT / "joints" / class_name / f"{video_path.stem}.npy"
        scores_path = OUTPUT_ROOT / "scores" / class_name / f"{video_path.stem}.npy"

        # Resume: bỏ qua clip đã lưu và có shape đúng.
        if joints_path.exists() and scores_path.exists():
            try:
                old_joints = np.load(joints_path, mmap_mode="r")
                old_scores = np.load(scores_path, mmap_mode="r")
                if (
                    old_joints.ndim == 4
                    and old_joints.shape[1:] == (2, 17, 3)
                    and old_scores.shape == old_joints.shape[:2] + (17,)
                ):
                    continue
            except Exception:
                pass

        try:
            joints, scores = extract_video(video_path)
            save_npy_atomic(joints_path, joints)
            save_npy_atomic(scores_path, scores)

            writer.writerow({
                "video": str(video_path),
                "status": "success",
                "frames": len(joints),
                "error": "",
            })
        except Exception as error:
            writer.writerow({
                "video": str(video_path),
                "status": "error",
                "frames": 0,
                "error": repr(error),
            })

        status_file.flush()

        if index % 10 == 0:
            elapsed = time.time() - started
            print(f"{index}/{len(videos_to_run)} video, {elapsed:.1f} giây")

print("Kết quả:", OUTPUT_ROOT)
print("Trạng thái:", STATUS_PATH)

In [ ]:
import csv
import os
import time

STATUS_PATH = OUTPUT_ROOT / "status.csv"

# Pilot trước; đổi thành None sau khi kiểm tra thành công.
MAX_VIDEOS = 50
videos_to_run = video_paths[:MAX_VIDEOS] if MAX_VIDEOS else video_paths
print(f"Sẽ xử lý {len(videos_to_run)} / {len(video_paths)} video")
print(f"Kết quả sẽ lưu dưới: {OUTPUT_ROOT}")


def save_npy_atomic(path, array):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(path.stem + ".tmp.npy")
    np.save(temp_path, array)
    os.replace(temp_path, path)


status_exists = STATUS_PATH.exists()
with open(STATUS_PATH, "a", newline="") as status_file:
    writer = csv.DictWriter(
        status_file,
        fieldnames=["video", "status", "frames", "error"],
    )
    if not status_exists:
        writer.writeheader()

    started = time.time()
    for index, video_path in enumerate(videos_to_run, start=1):
        class_name = video_path.parent.name
        joints_path = OUTPUT_ROOT / "joints" / class_name / f"{video_path.stem}.npy"
        scores_path = OUTPUT_ROOT / "scores" / class_name / f"{video_path.stem}.npy"

        # Resume safely: skip pairs already saved with expected shapes.
        if joints_path.exists() and scores_path.exists():
            try:
                old_joints = np.load(joints_path, mmap_mode="r")
                old_scores = np.load(scores_path, mmap_mode="r")
                if (
                    old_joints.ndim == 4
                    and old_joints.shape[1:] == (2, 17, 3)
                    and old_scores.shape == old_joints.shape[:2] + (17,)
                ):
                    continue
            except Exception:
                pass

        try:
            joints, scores = extract_video(video_path)
            save_npy_atomic(joints_path, joints)
            save_npy_atomic(scores_path, scores)
            writer.writerow({
                "video": str(video_path),
                "status": "success",
                "frames": len(joints),
                "error": "",
            })
        except Exception as error:
            writer.writerow({
                "video": str(video_path),
                "status": "error",
                "frames": 0,
                "error": repr(error),
            })

        status_file.flush()
        if index % 10 == 0:
            elapsed = time.time() - started
            print(f"{index}/{len(videos_to_run)} video, {elapsed:.1f} giây")

print("Hoàn tất batch.")
print("Trạng thái:", STATUS_PATH)
